# NuminaMath preprocessing for SFT

This notebook prepares NuminaMath for supervised fine-tuning:

1. configure HuggingFace caches and repository paths
2. load the dataset from local disk or HuggingFace
3. build a custom train/test split with a configurable fraction
4. extract the boxed answer and format prompt/completion records for `SFTTrainer`
5. save the processed dataset back to disk for reuse


## 1. Environment and paths

Set HuggingFace cache variables before importing `datasets` or `transformers`.

In [1]:
import os
from pathlib import Path


def resolve_project_root() -> Path:
    """Return the repository root regardless of whether the notebook runs from the root or notebooks/."""
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    return cwd


PROJECT_ROOT = resolve_project_root()
CACHE_ROOT = PROJECT_ROOT / ".cache" / "huggingface"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME", str(CACHE_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(CACHE_ROOT / "datasets"))
os.environ.setdefault("HF_HUB_CACHE", str(CACHE_ROOT / "hub"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(CACHE_ROOT / "transformers"))

DATASETS_DIR = PROJECT_ROOT / "datasets"
PROCESSED_DIR = DATASETS_DIR / "processed" / "numinamath_sft"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

HF_DATASET_ID = "AI-MO/NuminaMath-TIR"
RAW_LOCAL_DATASET_DIR = DATASETS_DIR

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CACHE_ROOT:   {CACHE_ROOT}")
print(f"DATASETS_DIR:  {DATASETS_DIR}")
print(f"PROCESSED_DIR: {PROCESSED_DIR}")

PROJECT_ROOT: /home/benjamin.deporte/GenAI_Math
CACHE_ROOT:   /home/benjamin.deporte/GenAI_Math/.cache/huggingface
DATASETS_DIR:  /home/benjamin.deporte/GenAI_Math/datasets
PROCESSED_DIR: /home/benjamin.deporte/GenAI_Math/datasets/processed/numinamath_sft


## 2. Load and split the dataset

Load the prepared dataset from disk when available, otherwise fall back to HuggingFace. The split is built from the full example pool so the train/test ratio is configurable.

In [2]:
from datasets import DatasetDict, concatenate_datasets, load_dataset, load_from_disk
import numpy as np


TRAIN_FRACTION = 0.99
SPLIT_SEED = 42
LOCAL_DATASET_CANDIDATES = [RAW_LOCAL_DATASET_DIR, RAW_LOCAL_DATASET_DIR / "numinamath", PROCESSED_DIR]


def load_numinamath_dataset(local_candidates: list[Path] | None = None, dataset_id: str = HF_DATASET_ID) -> DatasetDict:
    """Load NuminaMath from the first available local dataset directory, otherwise from HuggingFace."""
    candidates = local_candidates or LOCAL_DATASET_CANDIDATES
    for candidate in candidates:
        if candidate.exists() and (candidate / "dataset_dict.json").exists():
            print(f"Loading dataset from disk: {candidate}")
            loaded = load_from_disk(str(candidate))
            return loaded if isinstance(loaded, DatasetDict) else DatasetDict({"train": loaded})
    print(f"Loading dataset from HuggingFace: {dataset_id}")
    loaded = load_dataset(dataset_id)
    return loaded if isinstance(loaded, DatasetDict) else DatasetDict({"train": loaded})


def build_custom_split(dataset: DatasetDict, train_fraction: float = 0.99, seed: int = 42) -> DatasetDict:
    """Merge the available splits, then create a custom train/test split from the full pool."""
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction must be strictly between 0 and 1.")

    splits = [dataset[name] for name in dataset.keys()]
    full_dataset = splits[0] if len(splits) == 1 else concatenate_datasets(splits)
    split = full_dataset.train_test_split(test_size=1.0 - train_fraction, seed=seed, shuffle=True)
    return DatasetDict(train=split["train"], test=split["test"])


raw_dataset = load_numinamath_dataset()
custom_split = build_custom_split(raw_dataset, train_fraction=TRAIN_FRACTION, seed=SPLIT_SEED)

print(raw_dataset)
print(custom_split)
print(f"Train size: {len(custom_split['train'])}")
print(f"Test size:  {len(custom_split['test'])}")

Loading dataset from disk: /home/benjamin.deporte/GenAI_Math/datasets
DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 72441
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 99
    })
})
DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 71814
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages'],
        num_rows: 726
    })
})
Train size: 71814
Test size:  726


## 3. Boxed-answer extraction and record formatting

Extract the chain-of-thought from the solution text, pull the final answer from `\boxed{...}`, and format each example as a conversational prompt/completion pair.

In [3]:
import re
from typing import Any


BOXED_MARKER = r"\boxed{"


def extract_boxed(solution: str) -> str:
    """Extract the content of the first balanced \boxed{...} expression."""
    start = solution.find(BOXED_MARKER)
    if start == -1:
        return ""

    brace_index = start + len(BOXED_MARKER) - 1
    depth = 0
    for position in range(brace_index, len(solution)):
        if solution[position] == "{":
            depth += 1
        elif solution[position] == "}":
            depth -= 1
            if depth == 0:
                return solution[brace_index + 1 : position].strip()
    raise ValueError("Unmatched braces in \\boxed{} expression.")


def split_reasoning_and_answer(solution: str) -> tuple[str, str]:
    """Return the reasoning before the boxed answer and the boxed answer itself."""
    boxed = extract_boxed(solution)
    marker_index = solution.find(BOXED_MARKER)
    reasoning = solution if marker_index == -1 else solution[:marker_index].strip()
    reasoning = re.sub(r"\s+$", "", reasoning).strip()
    return reasoning, boxed


def extract_problem_and_solution(sample: dict[str, Any]) -> tuple[str, str]:
    """Normalize the problem and solution fields across NuminaMath variants."""
    problem = sample.get("problem")
    solution = sample.get("solution")

    if problem is None and isinstance(sample.get("messages"), list):
        messages = sample["messages"]
        if messages:
            problem = messages[0].get("content")
            if solution is None and len(messages) > 1:
                solution = messages[-1].get("content")

    if problem is None or solution is None:
        raise ValueError("Sample missing problem/solution content.")
    return problem, solution


def build_sft_record(sample: dict[str, Any]) -> dict[str, Any]:
    """Convert a raw sample into the conversational structure used by SFTTrainer."""
    problem, solution = extract_problem_and_solution(sample)
    reasoning, answer = split_reasoning_and_answer(solution)
    completion_text = f"<think>{reasoning}</think><answer>{answer}</answer>"
    return {
        "prompt": [{"role": "user", "content": problem}],
        "completion": [{"role": "assistant", "content": completion_text}],
        "problem": problem,
        "reasoning": reasoning,
        "answer": answer,
    }


def preprocess_dataset(dataset):
    """Map a dataset to the SFT conversational schema without removing the original columns."""
    return dataset.map(build_sft_record)


preview_sample = custom_split["train"][0]
preview_problem, preview_solution = extract_problem_and_solution(preview_sample)
preview_reasoning, preview_answer = split_reasoning_and_answer(preview_solution)

print(preview_problem)
print("--- reasoning preview ---")
print(preview_reasoning[:800])
print("--- answer preview ---")
print(preview_answer)

Do there exist 11 consecutive natural numbers whose sum is a perfect cube?
--- reasoning preview ---
To determine if there exist 11 consecutive natural numbers whose sum is a perfect cube, we can follow these steps:

1. **Formulate the Problem Mathematically:**
   Let the 11 consecutive natural numbers be \( n, n+1, n+2, \ldots, n+10 \).
   The sum of these numbers can be written as:
   \[
   S = n + (n+1) + (n+2) + \ldots + (n+10)
   \]

2. **Sum of an Arithmetic Sequence:**
   The sum of the first \(k\) natural numbers is given by:
   \[
   \text{Sum} = \frac{k}{2} (2a + (k-1)d)
   \]
   where \(a\) is the first term, \(d\) is the common difference, and \(k\) is the number of terms.
   For our problem, \(a = n\), \(d = 1\), and \(k = 11\).

   Therefore, the sum of the 11 consecutive numbers is:
   \[
   S = \frac{11}{2} (2n + 10) = 11(n + 5)
   \]

3. **Check if the Sum is a Perfect Cu
--- answer preview ---
\text{Yes}


## 4. Build and save the processed dataset

Create the conversational splits, save them to disk, and keep the saved artifact reloadable with `load_from_disk()`.

In [4]:
processed_dataset_path = PROCESSED_DIR / f"train_{int(TRAIN_FRACTION * 1000):03d}_seed_{SPLIT_SEED}"
processed_dataset_path.mkdir(parents=True, exist_ok=True)

processed_train = preprocess_dataset(custom_split["train"])
processed_test = preprocess_dataset(custom_split["test"])
processed_dataset = DatasetDict(train=processed_train, test=processed_test)

processed_dataset.save_to_disk(str(processed_dataset_path))
print(f"Saved processed dataset to: {processed_dataset_path}")
print(processed_dataset)

reloaded_dataset = load_from_disk(str(processed_dataset_path))
print(reloaded_dataset)
print(f"Reloaded train size: {len(reloaded_dataset['train'])}")
print(f"Reloaded test size:  {len(reloaded_dataset['test'])}")

Map:   0%|          | 0/71814 [00:00<?, ? examples/s]

Map:   0%|          | 0/726 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/71814 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/726 [00:00<?, ? examples/s]

Saved processed dataset to: /home/benjamin.deporte/GenAI_Math/datasets/processed/numinamath_sft/train_990_seed_42
DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages', 'prompt', 'completion', 'reasoning', 'answer'],
        num_rows: 71814
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages', 'prompt', 'completion', 'reasoning', 'answer'],
        num_rows: 726
    })
})
DatasetDict({
    train: Dataset({
        features: ['problem', 'solution', 'messages', 'prompt', 'completion', 'reasoning', 'answer'],
        num_rows: 71814
    })
    test: Dataset({
        features: ['problem', 'solution', 'messages', 'prompt', 'completion', 'reasoning', 'answer'],
        num_rows: 726
    })
})
Reloaded train size: 71814
Reloaded test size:  726


## 5. Validation preview

Inspect a few transformed rows and confirm the conversational schema is ready for `SFTTrainer`.

In [5]:
from pprint import pprint


sample_indices = np.random.default_rng(SPLIT_SEED).choice(
    len(processed_dataset["train"]),
    size=min(3, len(processed_dataset["train"])),
    replace=False,
).tolist()

for index in sample_indices:
    example = processed_dataset["train"][index]
    print("=" * 80)
    print(f"Index: {index}")
    print("Prompt:")
    pprint(example["prompt"])
    print("Completion:")
    pprint(example["completion"])
    print(f"Reasoning length: {len(example['reasoning'])}")
    print(f"Answer: {example['answer']}")

print("=" * 80)
print(f"Prompt/completion schema example: {processed_dataset['train'][0].keys()}")
print(f"Processed dataset path exists: {processed_dataset_path.exists()}")

Index: 47007
Prompt:
[{'content': 'On August 16, 2022, the airlock chamber of the Tianwen '
             'Experimental Module, the main exit channel for astronauts, made '
             "its first appearance. In order to understand the students' level "
             'of interest in this news, a certain high school used stratified '
             'sampling to select 36 students from three grades. Among them, 15 '
             'students were selected from the first grade, 12 students from '
             'the second grade, and there are a total of 900 students in the '
             'third grade. The total number of students in this high school is '
             '______.',
  'role': 'user'}]
Completion:
[{'content': '<think>To solve this problem, we need to determine the total '
             'number of students in the high school using the information '
             'given about the stratified sampling.\n'
             '\n'
             'The total number of students selected is 36, which inc